# Ticket 14: local_amount = 0 investigation

Split from #13/ADR 0006. `local_amount` reads exactly `0` on rows where
`debit_amount` or `credit_amount` is nonzero - a different shape than
#13's document-total broadcast. First check: does the `source IN
('automated', 'adjustment')` correlation mission 13 first noticed hold
up, or was it a hunch the way line count was for #13?

In [1]:
import duckdb  # type: ignore

con = duckdb.connect("../warehouse.duckdb", read_only=True)
FLAG = "local_amount = 0 AND (debit_amount != 0 OR credit_amount != 0)"
con.execute(f"SELECT COUNT(*) FROM stg_gl WHERE {FLAG}").fetchone()

(931,)

## Forced test case: 115020 / 205020

Where this pattern was first spotted. Any candidate predictor has to
explain this pair.

In [2]:
con.execute("""
    SELECT gl_account, document_type, source, COUNT(*) n,
           SUM((debit_amount!=0)::int) n_debit, SUM((credit_amount!=0)::int) n_credit,
           SUM((local_amount=0)::int) n_zero_local
    FROM stg_gl WHERE gl_account IN (115020, 205020)
    GROUP BY 1, 2, 3 ORDER BY 1, 2, 3
""").fetchdf()

,gl_account,document_type,source,n,n_debit,n_credit,n_zero_local
0,115020,IC,automated,24,24.0,0.0,24.0
1,205020,IC,automated,24,0.0,24.0,24.0


In [3]:
import csv
with open("../map_account.csv") as f:
    for r in csv.DictReader(f):
        if r["source_account"] in ("115020", "205020"):
            print(r)

{'source_account': '115020', 'target_account': '', 'status': 'unmapped', 'source_usage': 'live', 'account_role': 'clearing_pair', 'dq_flag': 'local_amount_zero_but_dr_cr_nonzero', 'pair_id': '115020_205020', 'notes': ''}
{'source_account': '205020', 'target_account': '', 'status': 'unmapped', 'source_usage': 'live', 'account_role': 'clearing_pair', 'dq_flag': 'local_amount_zero_but_dr_cr_nonzero', 'pair_id': '115020_205020', 'notes': ''}


## Testing every field issue #14 asked to check

precision = fraction of the candidate's own rows that are defective.
recall = fraction of the 931 defective rows the candidate covers.

In [ ]:
candidates = [
    ("document_type='IC'", "document_type = 'IC'"),
    ("document_type='SA'", "document_type = 'SA'"),
    ("source IN ('automated','adjustment')", "source IN ('automated','adjustment')"),
    ("account_role=clearing_pair (6 accts from map_account.csv)",
     "gl_account IN (115020,115021,115030,205020,205021,205030)"),
    ("currency='USD'", "currency = 'USD'"),
    ("exchange_rate=1.0", "exchange_rate = 1.0"),
    ("source_system IS NULL", "source_system IS NULL"),
    ("created_by IN ('IC_GENERATOR','SYSTEM')", "created_by IN ('IC_GENERATOR','SYSTEM')"),
    ("approver IS NULL", "approver IS NULL"),
]
rows = []
for label, cond in candidates:
    total = con.execute(f"SELECT COUNT(*) FROM stg_gl WHERE {cond}").fetchone()[0]
    hit = con.execute(f"SELECT COUNT(*) FROM stg_gl WHERE {cond} AND {FLAG}").fetchone()[0]
    rows.append((label, total, hit, round(100.0*hit/total, 2) if total else 0, round(100.0*hit/931, 2)))
import pandas as pd # type: ignore
pd.DataFrame(rows, columns=["candidate", "total", "hit", "precision_pct", "recall_pct"])

,candidate,total,hit,precision_pct,recall_pct
0,document_type='IC',860,680,79.07,73.04
1,document_type='SA',133981,251,0.19,26.96
2,"source IN ('automated','adjustment')",8203,931,11.35,100.00
3,account_role=clearing_pair (6 accts from map_a...,158,158,100.00,16.97
4,currency='USD',165426,931,0.56,100.00
5,exchange_rate=1.0,627424,931,0.15,100.00
6,source_system IS NULL,8661,931,10.75,100.00
7,"created_by IN ('IC_GENERATOR','SYSTEM')",8485,931,10.97,100.00
8,approver IS NULL,648801,931,0.14,100.00


No single field closes it the way `source = 'AB'` closed #13. Reversal
overlap and other structural fields checked separately below, all with
zero signal.

In [5]:
checks = {
    "reversal overlap (reference LIKE 'REV-%')":
        "SELECT COUNT(*) FROM stg_gl WHERE " + FLAG + " AND reference LIKE 'REV-%'",
    "is_manual breakdown":
        "SELECT is_manual, COUNT(*) FROM stg_gl WHERE " + FLAG + " GROUP BY 1",
    "predecessor_line_id populated":
        "SELECT COUNT(*) FROM stg_gl WHERE " + FLAG + " AND predecessor_line_id IS NOT NULL",
    "lettrage populated":
        "SELECT COUNT(*) FROM stg_gl WHERE " + FLAG + " AND lettrage IS NOT NULL",
}
for label, q in checks.items():
    print(label, "->", con.execute(q).fetchall())

reversal overlap (reference LIKE 'REV-%') -> [(0,)]
is_manual breakdown -> [(False, 931)]
predecessor_line_id populated -> [(0,)]
lettrage populated -> [(0,)]


## Splitting by per-account defect rate

Not "which fields correlate with the flagged rows" - which accounts
are defective on effectively every occurrence versus only rarely,
checked against every row for that account in the whole file, not just
the 931 flagged ones.

In [6]:
q = """
WITH accts AS (SELECT DISTINCT gl_account FROM stg_gl WHERE %s)
SELECT s.gl_account, COUNT(*) AS n_total, SUM((%s)::int) AS n_defect,
       ROUND(100.0*SUM((%s)::int)/COUNT(*), 2) AS pct_defect
FROM stg_gl s JOIN accts a USING (gl_account)
GROUP BY 1 ORDER BY pct_defect DESC
""" % (FLAG, FLAG, FLAG)
con.execute(q).fetchdf()

,gl_account,n_total,n_defect,pct_defect
0,205010,77,77.0,100.00
1,115020,24,24.0,100.00
2,115010,80,80.0,100.00
3,4810,1,1.0,100.00
4,115021,36,36.0,100.00
5,115030,18,18.0,100.00
6,205020,24,24.0,100.00
7,4800,1,1.0,100.00
8,205021,29,29.0,100.00
9,205030,27,27.0,100.00


## Group A: 10 accounts at 100.00%, zero exceptions

`115010`/`115020`/`115021`/`115030`/`205010`/`205020`/`205021`/`205030`
(solid sample sizes, 18-80 rows each) plus `4800`/`4810` (1 row each,
too thin to weigh the same). Every one of these reads `local_amount=0`
on every single occurrence where debit or credit is nonzero, regardless
of who created it or what document type. `115xxx` (asset) paired
against `205xxx` (liability) reads as an intercompany clearing
structure - the same shape `map_account.csv` already tags
`account_role=clearing_pair` for 6 of the 10.

In [7]:
GROUP_A = "(115010,115020,115021,115030,205010,205020,205021,205030,4800,4810)"
a = con.execute(f"SELECT COUNT(*) FROM stg_gl WHERE {FLAG} AND gl_account IN {GROUP_A}").fetchone()[0]
b = con.execute(f"SELECT COUNT(*) FROM stg_gl WHERE {FLAG} AND gl_account NOT IN {GROUP_A}").fetchone()[0]
print("Group A:", a, " Group B:", b, " sum:", a + b, " (want 931)")

Group A: 317  Group B: 614  sum: 931  (want 931)


In [8]:
con.execute(f"""
    SELECT DISTINCT gl_account, account_description, financial_statement_category
    FROM stg_gl WHERE gl_account IN {GROUP_A} ORDER BY gl_account
""").fetchdf()

,gl_account,account_description,financial_statement_category
0,4800,None,revenue
1,4810,None,revenue
2,115010,None,asset
3,115020,None,asset
4,115021,None,asset
5,115030,None,asset
6,205010,None,liability
7,205020,None,liability
8,205021,None,liability
9,205030,None,liability


## Group B: 28 accounts, 614 rows, no clean predictor

`gl_account` alone doesn't explain these - the same accounts return a
correct `local_amount` the overwhelming majority of the time. Narrowed
to `created_by IN ('SYSTEM', 'IC_GENERATOR')` (100% recall, only ~11%
precision) and checked whether a specific `gl_account` within that
narrower set is a clean predictor.

In [9]:
con.execute(f"""
    SELECT created_by, document_type,
           SUM(({FLAG})::int) AS n_defect, SUM((NOT ({FLAG}))::int) AS n_clean
    FROM stg_gl WHERE source_system IS NULL
    GROUP BY 1, 2 ORDER BY n_defect DESC, n_clean DESC
""").fetchdf()

,created_by,document_type,n_defect,n_clean
0,IC_GENERATOR,IC,680.0,180.0
1,SYSTEM,SA,251.0,6848.0
2,CLOSE_ENGINE,CL,0.0,176.0
3,SYSTEM,WE,0.0,110.0
4,SYSTEM,KR,0.0,100.0
5,SYSTEM,KZ,0.0,100.0
6,SYSTEM,OPENING_BALANCE,0.0,68.0
7,SYSTEM,WL,0.0,52.0
8,SYSTEM,DR,0.0,48.0
9,SYSTEM,DZ,0.0,48.0


In [10]:
con.execute(f"""
    SELECT fiscal_period, SUM(({FLAG})::int) n_defect, COUNT(*) n_total,
           ROUND(100.0*SUM(({FLAG})::int)/COUNT(*), 2) AS pct
    FROM stg_gl WHERE gl_account NOT IN {GROUP_A} AND created_by IN ('IC_GENERATOR','SYSTEM')
    GROUP BY 1 ORDER BY 1
""").fetchdf()

,fiscal_period,n_defect,n_total,pct
0,1,55.0,1309,4.20
1,2,40.0,622,6.43
2,3,40.0,586,6.83
3,4,48.0,730,6.58
4,5,39.0,729,5.35
5,6,41.0,675,6.07
6,7,54.0,566,9.54
7,8,45.0,553,8.14
8,9,47.0,675,6.96
9,10,50.0,576,8.68


Fiscal period 12 shows a mild skew (16.9% vs 5-9% for other periods)
but nowhere near a clean trigger. `is_anomaly=true` covers 35 of the
773 non-clearing rows across 7 different `anomaly_type` values -
scattered, not a cause.

**Conclusion: two mechanisms, not one.** Group A (10 accounts, 100%,
structural) reads as intentional - `local_amount` doesn't apply to
these accounts by design. Group B (28 accounts, 0.06%-9.72% each, 614
rows) is real but its trigger is unresolved after testing every field
issue #14 named. Both decisions are recorded in
`docs/missions/14-local-amount-zero-investigation.md`, not assumed
here.